# max-back-tied-half — faded example 1: Build the maximum_back0 Mask Using the Half-Mass Convention

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `max-back-tied-half`. The last cell reports your progress on the `Backprop: max_back with tied half-mass` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: max_back with tied half-mass` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`max-back-tied-half`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "max-back-tied-half"
DD_SUBTOPIC = "Backprop: max_back with tied half-mass"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The gradient of `maximum(x, y)` with respect to `x` is `grad_out` multiplied by a mask: 1 where `x > y`, 0 where `x < y`, and 0.5 where `x == y`. This 50/50 tie-split is the only symmetric choice that conserves gradient mass across both inputs at tie positions.

## Faded exercise 1

Complete `maximum_back0` below. The function body is given except for the single blanked line that constructs the mask using the half-mass convention.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t

def maximum_back0(grad_out, x, y):
    mask = (x > y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
    return grad_out * mask

# Exercise it
x = t.tensor([3.0, 1.0, 2.0, 4.0])
y = t.tensor([1.0, 4.0, 2.0, 3.0])
grad_out = t.ones(4)
g0 = maximum_back0(grad_out, x, y)
print(f"g0: {g0.tolist()}  (expected [1.0, 0.0, 0.5, 1.0])")


import torch as t

def _test():
    x = t.tensor([3.0, 1.0, 2.0, 4.0])
    y = t.tensor([1.0, 4.0, 2.0, 3.0])
    grad_out = t.ones(4)

    g0 = maximum_back0(grad_out, x, y)

    expected = t.tensor([1.0, 0.0, 0.5, 1.0])
    assert t.allclose(g0, expected, atol=1e-6), f"Got {g0.tolist()}, expected {expected.tolist()}"

    # Test with non-unit grad_out
    grad2 = t.tensor([2.0, 3.0, 4.0, 5.0])
    g0_2 = maximum_back0(grad2, x, y)
    expected2 = t.tensor([2.0, 0.0, 2.0, 5.0])
    assert t.allclose(g0_2, expected2, atol=1e-6)

    # Mass conservation: g0 + g1 == grad_out
    def maximum_back1(grad_out, x, y):
        mask = (x < y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
        return grad_out * mask
    g1 = maximum_back1(grad_out, x, y)
    assert t.allclose(g0 + g1, grad_out, atol=1e-6)


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def maximum_back0(grad_out, x, y):
    mask = (x > y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
    return grad_out * mask

# Exercise it
x = t.tensor([3.0, 1.0, 2.0, 4.0])
y = t.tensor([1.0, 4.0, 2.0, 3.0])
grad_out = t.ones(4)
g0 = maximum_back0(grad_out, x, y)
print(f"g0: {g0.tolist()}  (expected [1.0, 0.0, 0.5, 1.0])")
```
</details>